In [ ]:
import os

# Load configuration from YAML file
config = {
    "model_name": "llama3-70b-8192",
    # "model_name": "llama3-8b-8192",
    # "model_name": "llama-3.1-70b-versatile",
    # "model_name": "llama-3.1-8b-instant",
    # "model_name": "gemma2-9b-it",
}


In [ ]:
import json
import yaml
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.schema.output_parser import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import JsonOutputParser
from langchain.output_parsers import YamlOutputParser

# Define prompt strings as constants
DESCRIPTION_PROMPT = [
    ("system", """Given the following JSON example for a task type:
{raw_example}

Provide a concise description of the task type, including the format and style of the output.

Format your response as follows:
Task Description: [Your description here]
""")
]

BRIEFS_PROMPT = [
    ("system", """Given the task type description, generate descriptions for {generating_batch_size} new 
examples with detailed attributes based on this task type. But don't provide any detailed task output.

Format your response as a valid YAML object with a single key
'new_example_briefs' containing a YAML array of {generating_batch_size} objects, each
with a 'example_brief' field.
"""),
    ("user", """Task Description:

{description}

""")
]

EXAMPLES_FROM_BRIEFS_PROMPT = [
    ("system", """Given the task type description, brief descriptions for new examples, 
and JSON example, generate 3 more input/output examples for this task type,
strictly based on the brief descriptions. Ensure that the new examples are
consistent with the brief descriptions and do not introduce any new information
not present in the briefs.

Format your response as a valid JSON object with a single key 'examples' 
containing a JSON array of {generating_batch_size} objects, each with 'input' and 'output' fields.
"""),
    ("user", """Task Description:

{description}

New Example Briefs: 

{new_example_briefs}

Example:

{raw_example}

""")
]

EXAMPLES_PROMPT = [
    ("system", """Given the task type description, and input/output example, generate {generating_batch_size}
new input/output examples for this task type.

Format your response as a valid JSON object with a single key 'examples' 
containing a JSON array of {generating_batch_size} objects, each with 'input' and 'output' fields.
"""),
    ("user", """Task Description:

{description}

Example:

{raw_example}

""")
]

class TaskDescriptionGenerator:
    def __init__(self, model):        
        self.description_prompt = ChatPromptTemplate.from_messages(DESCRIPTION_PROMPT)
        self.briefs_prompt = ChatPromptTemplate.from_messages(BRIEFS_PROMPT)
        self.examples_from_briefs_prompt = ChatPromptTemplate.from_messages(EXAMPLES_FROM_BRIEFS_PROMPT)
        self.examples_prompt = ChatPromptTemplate.from_messages(EXAMPLES_PROMPT)

        json_model = model.bind(response_format={"type": "json_object"})

        output_parser = StrOutputParser()
        json_parse = JsonOutputParser()

        self.description_chain = self.description_prompt | model | output_parser
        self.briefs_chain = self.briefs_prompt | model | output_parser
        self.examples_from_briefs_chain = self.examples_from_briefs_prompt | json_model | json_parse
        self.examples_chain = self.examples_prompt | json_model | json_parse

        self.chain = (
            RunnablePassthrough.assign(raw_example = lambda x: json.dumps(x["example"], ensure_ascii=False))
            | RunnablePassthrough.assign(description = self.description_chain)
            | {
                "description": lambda x: x["description"],
                "examples_from_briefs": RunnablePassthrough.assign(new_example_briefs = lambda x: self.briefs_chain.invoke(x)) | self.examples_from_briefs_chain,
                "examples": self.examples_chain
            }
            | RunnablePassthrough.assign(
                additional_examples=lambda x: (
                    list(x["examples_from_briefs"]["examples"])
                    + list(x["examples"]["examples"])
                )
            )
        )

    def process(self, input_str, generating_batch_size=3):
        try:
            # Parse input string to a dictionary
            example_dict = json.loads(input_str) if input_str.startswith('{') else yaml.safe_load(input_str)
            if not isinstance(example_dict, dict) or 'input' not in example_dict or 'output' not in example_dict:
                raise ValueError("Invalid input format. Expected an object with 'input' and 'output' fields.")

            # Move the original content to a key named 'example'
            input_dict = {"example": example_dict, "generating_batch_size": generating_batch_size}

            # Invoke the chain with the parsed input dictionary
            result = self.chain.invoke(input_dict)
            return result

        except Exception as e:
            raise RuntimeError(f"An error occurred during processing: {str(e)}")

In [ ]:
import gradio as gr

def process_json(input_json, model_name, generating_batch_size=3):
    try:
        model = ChatOpenAI(model=model_name)
        generator = TaskDescriptionGenerator(model)
        result = generator.process(input_json, generating_batch_size)
        description = result["description"]
        examples = [[example["input"], example["output"]] for example in result["additional_examples"]]
        return description, examples
    except Exception as e:
        raise gr.Error(f"An error occurred: {str(e)}")

demo = gr.Interface(
    fn=process_json,
    inputs=[
        gr.Textbox(label="Input JSON"),
        gr.Dropdown(label="Model Name", choices=["llama3-70b-8192", "llama3-8b-8192", "llama-3.1-70b-versatile", "llama-3.1-8b-instant", "gemma2-9b-it"], value="llama3-70b-8192"),
        gr.Slider(label="Generating Batch Size", value=3, minimum=1, maximum=10, step=1)
    ],
    outputs=[
        gr.Textbox(label="Description"),
        gr.DataFrame(label="Examples", headers=["Input", "Output"])
    ],
    title="Task Description Generator",
    description="Enter a JSON object with 'input' and 'output' fields to generate a task description and additional examples.",
    allow_flagging="manual",
    flagging_callback=gr.CSVLogger()
)

if __name__ == "__main__":
    demo.launch()